[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xiptos/is_notes/blob/main/linear_regression_exercise_insurance.ipynb)

# Linear regression from scratch: medical insurance costs

This is an **advanced** exercise. You will:

* **Part A:** implement linear regression yourself with NumPy (cost function, gradient, gradient descent and the normal equation) and check it against scikit-learn;
* **Part B:** use your implementation on a real dataset of [medical insurance charges](https://github.com/stedy/Machine-Learning-with-R-datasets/blob/master/insurance.csv), which needs categorical encoding and feature scaling;
* **Part C:** engineer features (interactions, non-linear terms, a log-transformed target) guided by the plots;
* **Part D:** control overfitting with **regularisation** (Ridge and Lasso) and choose the hyperparameter with **cross-validation**;
* **Part E:** evaluate on the test set once and interpret the final model.

### How this notebook works

* Cells with `...` or `# TODO` are for you to complete.
* **Predict first** cells: write what you *expect* **before** running the code. You are graded on your reasoning, not on whether the guess was right.
* **Explain** cells: answer in your own words, using **your own numbers**. Your data subset and split depend on your student number, so your results differ from your colleagues'.
* `check(...)` cells test your work. Do not move on until they pass.
* Part A may only use NumPy: `sklearn` is used only to *compare* your results.

## 0. Your student number

In [ ]:
STUDENT_ID = None  # TODO: replace with your student number, e.g. 12345

assert isinstance(STUDENT_ID, int), "Set STUDENT_ID to your student number (an integer)."
SEED = STUDENT_ID % (2**32)

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


def check(condition, message):
    if not condition:
        raise AssertionError("❌ " + message)
    print("✅", message)

# Part A: linear regression with NumPy

With $m$ examples stacked as the rows of a design matrix $X \in \mathbb{R}^{m \times (n+1)}$ (the first column is all ones, $x_0^{(i)} = 1$), the hypothesis for all examples at once is

$$\hat{y} = X\theta$$

the cost function is

$$J(\theta) = \frac{1}{2m} \sum_{i=1}^{m} \left(h_\theta(x^{(i)}) - y^{(i)}\right)^2 = \frac{1}{2m}(X\theta - y)^T(X\theta - y)$$

and its gradient is

$$\nabla_\theta J = \frac{1}{m} X^T (X\theta - y)$$

Gradient descent repeats $\theta := \theta - \alpha \nabla_\theta J$. The **normal equation** gives the minimum directly by solving $X^T X \theta = X^T y$.

Implement the functions below **without loops over the examples** (use matrix operations).

In [ ]:
def add_bias(X):
    # Return X with a first column of ones. X has shape (m, n); the result (m, n+1).
    ...  # TODO


def predict(Xb, theta):
    ...  # TODO


def compute_cost(Xb, y, theta):
    ...  # TODO


def compute_gradient(Xb, y, theta):
    ...  # TODO


def gradient_descent(Xb, y, alpha=0.1, n_iters=1000):
    # Start from theta = 0. Return the final theta and the list of costs, one per iteration.
    theta = np.zeros(Xb.shape[1])
    history = []
    for _ in range(n_iters):
        ...  # TODO
    return theta, history


def normal_equation(Xb, y):
    # Hint: np.linalg.solve is faster and more accurate than computing an inverse
    ...  # TODO

### A.1 Testing on a toy problem

The next cell checks your functions on a small example where the answer is known. It also checks your gradient with a **numerical approximation**: for a small $\varepsilon$,

$$\frac{\partial J}{\partial \theta_j} \approx \frac{J(\theta + \varepsilon e_j) - J(\theta - \varepsilon e_j)}{2\varepsilon}$$

This is a standard technique to find bugs in hand-written gradients.

In [ ]:
Xt = np.array([[1.0], [2.0], [3.0]])
yt = np.array([3.0, 5.0, 7.0])          # y = 1 + 2x
Xtb = add_bias(Xt)

check(Xtb.shape == (3, 2) and np.all(Xtb[:, 0] == 1), "add_bias adds a first column of ones")
check(np.isclose(compute_cost(Xtb, yt, np.array([1.0, 2.0])), 0), "cost is 0 for the perfect theta")
check(np.isclose(compute_cost(Xtb, yt, np.zeros(2)), (9 + 25 + 49) / 6), "cost for theta = 0")

theta_rand = np.random.RandomState(SEED).randn(2)
eps = 1e-6
numeric = np.array([(compute_cost(Xtb, yt, theta_rand + eps * e) - compute_cost(Xtb, yt, theta_rand - eps * e)) / (2 * eps)
                    for e in np.eye(2)])
check(np.allclose(compute_gradient(Xtb, yt, theta_rand), numeric, atol=1e-5), "gradient matches the numerical approximation")

check(np.allclose(normal_equation(Xtb, yt), [1, 2]), "normal equation finds theta = [1, 2]")
theta_gd, hist = gradient_descent(Xtb, yt, alpha=0.1, n_iters=5000)
check(np.allclose(theta_gd, [1, 2], atol=1e-3), "gradient descent converges to theta = [1, 2]")
check(all(b <= a + 1e-12 for a, b in zip(hist, hist[1:])), "the cost never increases")

**Explain:** The gradient formula has $\frac{1}{m}$ and the cost has $\frac{1}{2m}$. Where does the $2$ go? Would gradient descent still find the same $\theta$ if we removed the $\frac{1}{2m}$ from the cost (and the $\frac{1}{m}$ from the gradient)? What would change in practice?

**Your answer:**



# Part B: the insurance dataset

### B.1 Download and load

In [ ]:
import os
import urllib.request

URL = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv"
FILE = "insurance.csv"
if not os.path.exists(FILE):
    urllib.request.urlretrieve(URL, FILE)

raw = pd.read_csv(FILE)
# Each student works on a different random 90% of the people
df = raw.sample(frac=0.9, random_state=SEED).reset_index(drop=True)
df.head()

Each row is a person insured in the USA. `charges` is the target: the yearly medical costs billed by the insurance, in dollars.

### B.2 Exploration

Explore the dataset (shape, types, missing values, descriptive statistics), then make **three** plots:

1. the distribution of `charges` ([`histplot`](https://seaborn.pydata.org/generated/seaborn.histplot.html));
2. `charges` vs `age`, coloured by `smoker` (`hue="smoker"`);
3. `charges` vs `bmi`, coloured by `smoker`, with a vertical line at `bmi = 30` (the threshold for obesity).

In [ ]:
# TODO: shape, info, missing values, describe

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
# TODO: plot 1 on axes[0], plot 2 on axes[1], plot 3 on axes[2]  (seaborn functions accept ax=...)

**Explain:** Describe what you see in each of the three plots. In particular:
1. Is `charges` symmetric? What does that mean for the squared error?
2. In the `age` plot you should see roughly three "bands". Which groups of people do you think form each band?
3. Does `bmi` affect smokers and non-smokers in the same way?

Your answers here should guide the features you create in Part C.

**Your answer:**



### B.3 Encoding

Complete `encode` so that it turns a DataFrame with the original columns into a **numeric** DataFrame:

* `sex_male` = 1 if `sex` is `"male"`, else 0;
* `smoker_yes` = 1 if `smoker` is `"yes"`, else 0;
* one dummy column per region, dropping `"northeast"` (the region part is given).

We use a function instead of calling `pd.get_dummies` on the whole dataset because we will need to encode **new people** later, one at a time, and they must get exactly the same columns.

In [ ]:
REGIONS = ["northeast", "northwest", "southeast", "southwest"]


def encode(data):
    out = pd.DataFrame(index=data.index)
    out["age"] = data["age"].astype(float)
    out["bmi"] = data["bmi"].astype(float)
    out["children"] = data["children"].astype(float)
    out["sex_male"] = ...    # TODO
    out["smoker_yes"] = ...  # TODO
    for region in REGIONS[1:]:
        out["region_" + region] = (data["region"] == region).astype(float)
    return out


X_all = encode(df)
y_all = df["charges"].to_numpy()
X_all.head()

In [ ]:
check(X_all.shape[1] == 8 and all(X_all.dtypes == float), "8 numeric columns")
check(X_all["smoker_yes"].sum() == (df["smoker"] == "yes").sum(), "smoker_yes is correct")

### B.4 Split

Split into 80% train and 20% test with `random_state=SEED`. From now until Part E, **the test set is locked**: do not use `X_test` or `y_test` for anything.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = ...  # TODO

### B.5 Why feature scaling matters

**Predict first:** `age` goes from 18 to 64 and the dummy variables are 0 or 1. If you run gradient descent on the **unscaled** data, what do you expect to happen with a large learning rate ($\alpha = 0.01$)? And with a very small one ($\alpha = 10^{-4}$)?

**Your answer:**



Run the experiment. The warnings you may get are part of the answer.

In [ ]:
Xb_train_raw = add_bias(X_train.to_numpy())

plt.figure(figsize=(8, 4))
for alpha in [1e-2, 1e-3, 1e-4]:
    _, hist = gradient_descent(Xb_train_raw, y_train, alpha=alpha, n_iters=200)
    plt.plot(hist, label=f"alpha = {alpha}")
plt.yscale("log")
plt.xlabel("iteration")
plt.ylabel("cost J")
plt.legend()
plt.title("Gradient descent on unscaled features")

Now implement **standardisation**: for every column $j$, $x_j := \frac{x_j - \mu_j}{\sigma_j}$.

The code below has a **bug** that a lot of people make. It runs without errors and gives reasonable-looking numbers, but it is wrong. Find it, fix it, and explain it below.

In [ ]:
def standardize(X_train, X_test):
    X_train = np.asarray(X_train, dtype=float)
    X_test = np.asarray(X_test, dtype=float)
    mu_train, sigma_train = X_train.mean(axis=0), X_train.std(axis=0)
    mu_test, sigma_test = X_test.mean(axis=0), X_test.std(axis=0)
    return (X_train - mu_train) / sigma_train, (X_test - mu_test) / sigma_test, mu_train, sigma_train


Xs_train, Xs_test, mu, sigma = standardize(X_train, X_test)

In [ ]:
check(np.allclose(Xs_train.mean(axis=0), 0) and np.allclose(Xs_train.std(axis=0), 1), "training features have mean 0 and std 1")
check(np.allclose(Xs_test * sigma + mu, X_test.to_numpy()), "test features are scaled with the training statistics")

**Explain:** What was the bug? Why is it wrong, even though the test set gets mean 0 and standard deviation 1 either way? (Think about a test set with a single person.)

**Your answer:**



### B.6 Training on the scaled data

Train with **your** gradient descent on the scaled training set (choose $\alpha$ and the number of iterations, and plot the cost curve), and also with your normal equation. Then compare with scikit-learn's `LinearRegression` trained on the same scaled data.

In [ ]:
from sklearn.linear_model import LinearRegression

Xsb_train = add_bias(Xs_train)

theta_gd, hist = ...  # TODO
theta_ne = ...        # TODO

# TODO: plot the cost curve (hist)

sk = LinearRegression().fit(Xs_train, y_train)
theta_sk = np.r_[sk.intercept_, sk.coef_]

pd.DataFrame({"gradient descent": theta_gd, "normal equation": theta_ne, "sklearn": theta_sk},
             index=["intercept"] + list(X_train.columns))

In [ ]:
check(np.allclose(theta_ne, theta_sk), "normal equation matches sklearn")
check(np.allclose(theta_gd, theta_sk, rtol=1e-3, atol=1), "gradient descent matches sklearn")

**Explain:**
1. Why is the intercept equal to the mean of `y_train` (check it)?
2. Because the inputs are standardised, the coefficients can now be compared with each other. Which three features matter most? Does that agree with the plots in B.2?
3. The normal equation needs no $\alpha$ and no iterations. Why do we still learn gradient descent? (Think about $n = 10^5$ features, or $m = 10^9$ examples.)

**Your answer:**



### B.7 Cross-validated baseline

To compare models without touching the test set we use **5-fold cross-validation** on the training set. The helper below is given: it takes any scikit-learn model (or pipeline), and returns the mean validation RMSE in dollars.

The `target` argument lets you train on a transformed target (e.g. `np.log`) while still measuring the error in dollars: predictions are converted back with `inverse` before computing the RMSE.

In [ ]:
from sklearn.model_selection import KFold
from sklearn.base import clone

kfold = KFold(n_splits=5, shuffle=True, random_state=SEED)


def cv_rmse(model, X, y, target=None, inverse=None):
    X, y = np.asarray(X, dtype=float), np.asarray(y, dtype=float)
    errors = []
    for train_idx, val_idx in kfold.split(X):
        m = clone(model)
        y_fit = y[train_idx] if target is None else target(y[train_idx])
        m.fit(X[train_idx], y_fit)
        pred = m.predict(X[val_idx])
        if inverse is not None:
            pred = inverse(pred)
        errors.append(np.sqrt(np.mean((pred - y[val_idx]) ** 2)))
    return np.mean(errors)


comparison = {"B: linear, 8 features": cv_rmse(LinearRegression(), X_train, y_train)}
comparison

# Part C: feature engineering

Use what you observed in B.2 to create new features. Complete `engineer` so that it adds **at least**:

* `age2`: age squared (given as an example);
* `obese`: 1 if `bmi >= 30`, else 0;
* `smoker_obese`: the product of `smoker_yes` and `obese` (an **interaction**);
* one more feature of your own choice that you think could help, justified below.

**Predict first:** Which of these features do you expect to reduce the error the most, and why?

**Your answer:**



In [ ]:
def engineer(X):
    X = X.copy()
    X["age2"] = X["age"] ** 2
    # TODO
    return X


XE_train = engineer(X_train)
comparison["C: engineered features"] = cv_rmse(LinearRegression(), XE_train, y_train)
comparison

In [ ]:
check({"age2", "obese", "smoker_obese"} <= set(XE_train.columns), "required features exist")
check(comparison["C: engineered features"] < 0.8 * comparison["B: linear, 8 features"], "engineered features reduce the CV RMSE by more than 20%")

To find which feature helped most, run an **ablation**: remove one engineered feature at a time and measure how much the CV RMSE gets worse.

In [ ]:
engineered = [c for c in XE_train.columns if c not in X_train.columns]
ablation = {}
for feature in engineered:
    ablation[feature] = ...  # TODO: CV RMSE without this feature

pd.Series(ablation).sort_values(ascending=False)

**Explain:** Was your prediction right? Justify the extra feature you added and say whether the ablation shows that it was useful.

**Your answer:**



### C.2 Log-transformed target

Because `charges` is very skewed, a common trick is to model $\log(\text{charges})$ instead. Use `cv_rmse` with `target=np.log` and `inverse=np.exp` on the engineered features.

**Predict first:** Will the RMSE **in dollars** go down or up? (Careful: think about which errors the squared loss focuses on in each case.)

**Your answer:**



In [ ]:
comparison["C: engineered + log target"] = ...  # TODO
pd.Series(comparison)

**Explain:** What happened, and why? Give one situation in which the log model would be the better choice even if its RMSE is worse.

**Your answer:**



# Part D: regularisation

Instead of choosing features by hand, we can generate **all** degree-2 polynomial features (squares and pairwise products) and let regularisation control the complexity:

* **Ridge** minimises $J(\theta) + \lambda \sum_{j\geq1} \theta_j^2$;
* **Lasso** minimises $J(\theta) + \lambda \sum_{j\geq1} |\theta_j|$.

In scikit-learn $\lambda$ is called `alpha` (not to be confused with the learning rate!).

Complete the pipeline: [`PolynomialFeatures`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html) with `degree=2` and `include_bias=False`, then a [`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html), then the regressor.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge, Lasso
from sklearn.exceptions import ConvergenceWarning
import warnings

# Lasso warns a lot for very small lambdas; we accept that here
warnings.filterwarnings("ignore", category=ConvergenceWarning)


def poly_model(regressor):
    return make_pipeline(...)  # TODO


n_poly = PolynomialFeatures(degree=2, include_bias=False).fit(X_train).n_output_features_
print("number of polynomial features:", n_poly)

comparison["D: polynomial, no regularisation"] = cv_rmse(poly_model(LinearRegression()), X_train, y_train)
pd.Series(comparison)

**Explain:** Why must the `StandardScaler` be *inside* the pipeline (and therefore be refitted in every fold), instead of scaling `X_train` once before calling `cv_rmse`? Why do Ridge and Lasso need the features to be scaled, when plain `LinearRegression` does not?

**Your answer:**



### D.1 Choosing λ with cross-validation

For Ridge and for Lasso, compute the CV RMSE for every value in `lambdas`, and plot both curves on the same figure (log scale on the x axis). Pick the best λ of each.

For Lasso use `Lasso(alpha=lam, max_iter=50_000)`.

In [ ]:
lambdas = np.logspace(-2, 4, 25)

ridge_scores = ...  # TODO: list of CV RMSEs, one per lambda
lasso_scores = ...  # TODO

# TODO: plot

best_ridge = lambdas[np.argmin(ridge_scores)]
best_lasso = lambdas[np.argmin(lasso_scores)]
comparison[f"D: polynomial + Ridge (λ={best_ridge:.3g})"] = min(ridge_scores)
comparison[f"D: polynomial + Lasso (λ={best_lasso:.3g})"] = min(lasso_scores)
pd.Series(comparison)

**Explain:** Describe the shape of the curves. What is happening to the model on the far left (small λ) and on the far right (large λ) of the plot? Use the words *bias* and *variance*.

**Your answer:**



### D.2 What did Lasso keep?

Fit the Lasso pipeline with the best λ on the whole training set and list the features whose coefficient is **not** zero, sorted by absolute value. Feature names can be obtained from the pipeline with `model[0].get_feature_names_out(X_train.columns)`.

In [ ]:
lasso = poly_model(Lasso(alpha=best_lasso, max_iter=50_000)).fit(X_train, y_train)

names = ...  # TODO
coefs = pd.Series(..., index=names)  # TODO
kept = ...  # TODO: non-zero coefficients sorted by absolute value (largest first)

print(f"Lasso kept {len(kept)} of {len(coefs)} features")
kept

**Explain:** Compare the top features kept by Lasso with the ones you engineered by hand in Part C. Did Lasso "discover" the same interactions? Why can it not discover `obese` (bmi ≥ 30) exactly? You may find features such as `smoker_yes^2` in the list: why is that feature useless, and what problem does it create?

**Your answer:**



# Part E: final evaluation

Choose **one** final model based **only** on the cross-validation results in `comparison`. Train it on the whole training set and evaluate it on the test set, **once**. Report RMSE, MAE and $R^2$.

In [ ]:
from sklearn import metrics

print(pd.Series(comparison).sort_values())

final_model = ...  # TODO: your choice, fitted on the training set
final_pred = ...   # TODO: predictions for the test set (in dollars!)

final = {
    "RMSE": np.sqrt(metrics.mean_squared_error(y_test, final_pred)),
    "MAE": metrics.mean_absolute_error(y_test, final_pred),
    "R2": metrics.r2_score(y_test, final_pred),
}
final

Plot the residuals of the test set against the predictions, coloured by `smoker`.

In [ ]:
# TODO

**Explain:** Is the test RMSE close to the CV RMSE of the chosen model? What does it mean if it is much higher? Which people does the model still get wrong (look at the large residuals)?

**Your answer:**



### E.1 Using the model

Use your final model to answer: how much more does the model predict a **40-year-old male smoker** from the southeast with 1 child pays per year, compared with an otherwise identical **non-smoker**, when the BMI is **25**? And when the BMI is **35**?

The DataFrame of people is given. Remember that they must go through **the same transformations** as the training data.

In [ ]:
people = pd.DataFrame({
    "age": [40, 40, 40, 40],
    "sex": ["male"] * 4,
    "bmi": [25, 25, 35, 35],
    "children": [1] * 4,
    "smoker": ["no", "yes", "no", "yes"],
    "region": ["southeast"] * 4,
})

people["predicted charges"] = ...  # TODO
people

**Explain:** What is the extra cost of smoking at BMI 25 and at BMI 35? Why is it different, and which feature(s) of your model make it different? Would the basic 8-feature model from Part B give different extra costs for the two BMIs?

**Your answer:**



# Summary

Run the cell below and do not edit it. Your teacher can reproduce these numbers from your student number. It also shows the **question you must answer** in the cell after it.

In [ ]:
QUESTIONS = [
    "Using your plots from B.2 and your ablation from C, explain why a model with the smoker_obese interaction beats one with smoker_yes and bmi as separate features.",
    "Explain, with your numbers from C.2, why fitting log(charges) changed the RMSE in dollars the way it did, and when you would still prefer the log model.",
    "Using your curves from D.1, explain what would go wrong if you had chosen λ by looking at the test set instead of using cross-validation.",
    "Your gradient descent from Part A needed scaled features. Using the plot from B.5, explain why, and what the cost surface looks like with and without scaling.",
]
print("Student:", STUDENT_ID)
print(pd.Series(comparison).round(1).to_string())
print("Test:", {k: round(v, 3) for k, v in final.items()})
print("\nYour question:", QUESTIONS[STUDENT_ID % len(QUESTIONS)])

**Your answer:**



### AI usage declaration

If you used an AI tool (ChatGPT, Copilot, Gemini, Claude, ...) in this exercise, say **which one**, **for which sections**, and **what you asked it**. You may be asked to explain or change any cell of this notebook in class.

**Your answer:**

